# M02 — Public-Data Ingestion and Audit

This notebook reproduces the M02 Federal Reserve GSW data pipeline in a fresh Google Colab runtime. It downloads and audits market inputs only; it does not calculate bond prices, VaR, Expected Shortfall, or stress losses.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JoyWu-302121/market_risk.git'
PROJECT_DIR = Path('/content/market_risk') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB and not (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
elif IN_COLAB:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements-colab.txt'], check=True)
src_path = str(PROJECT_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f'Project directory: {PROJECT_DIR}')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

## Run the primary GSW pipeline

The raw source bytes, metadata, normalized curve, and audit report are written to the temporary Colab filesystem under `data/`.

In [ ]:
import yaml
from bond_risk.data.pipeline import run_gsw_pipeline

OUTPUT_ROOT = PROJECT_DIR / 'data'
configuration = yaml.safe_load((PROJECT_DIR / 'configs/data_sources.yaml').read_text())
gsw_configuration = configuration['gsw']
gsw_result = run_gsw_pipeline(
    OUTPUT_ROOT,
    analysis_start=gsw_configuration['analysis_start'],
    required_tenors=gsw_configuration['required_tenors_years'],
    source_url=gsw_configuration['url'],
    stale_after_days=gsw_configuration['stale_after_calendar_days'],
)
gsw_result['audit']

## Verify the accepted M02 invariants

In [ ]:
audit = gsw_result['audit']
assert audit['status'] != 'FAIL', audit['failures']
assert audit['duplicate_date_tenor_rows'] == 0
assert audit['invalid_yield_count'] == 0
for tenor in ('3', '7', '15'):
    assert audit['required_tenor_coverage'][tenor]['latest_available']
    assert audit['required_tenor_coverage'][tenor]['observed_ratio'] == 1.0
print('M02 GSW acceptance checks: PASS')

## Inspect normalized observations

The normalized table retains source percentage points and the decimal continuously compounded yield used by future models. Missing observations remain explicit.

In [ ]:
import pandas as pd
from IPython.display import display

curve = pd.read_csv(gsw_result['processed_path'], parse_dates=['observation_date'])
display(curve.tail(12))
display(pd.DataFrame(audit['required_tenor_coverage']).T)

## Optional: download FRED proxies

Add `FRED_API_KEY` through the Colab Secrets panel. The key is never written to project files or metadata. If the secret is unavailable, this section is skipped without affecting the primary GSW milestone.

In [ ]:
FRED_API_KEY = None
if IN_COLAB:
    try:
        from google.colab import userdata
        FRED_API_KEY = userdata.get('FRED_API_KEY')
    except Exception:
        FRED_API_KEY = None

if FRED_API_KEY:
    from bond_risk.data.pipeline import run_fred_pipeline

    fred_results = {
        series_id: run_fred_pipeline(
            OUTPUT_ROOT,
            series_id=series_id,
            api_key=FRED_API_KEY,
            observation_start='2005-01-01',
        )
        for series_id in ('DFF', 'DGS3MO')
    }
    print({key: value['audit']['status'] for key, value in fred_results.items()})
else:
    print('FRED_API_KEY is unavailable; optional FRED retrieval was skipped.')

## M02 completion boundary

M02 ends after source versioning, normalization, and data-quality auditing. M03 will construct discount factors, interpolate log discount factors, value the synthetic zero-coupon portfolio, and verify DV01 invariants.